# 02 — Data Quality & Cleaning
Formal quality checks, unit conversion (Kelvin→Celsius, Pa→hPa, m→mm), and derived fields.
Output is saved to `data/processed/` for use in later notebooks.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from data.load_clean import load_raw, clean_and_convert, quality_report

pd.set_option('display.max_columns', None)

In [2]:
raw = load_raw('../data/raw/jabalpur_weather_2024_2025.csv')
report = quality_report(raw.assign(valid_time=pd.to_datetime(raw['valid_time'])))
for k, v in report.items():
    print(f'{k}: {v}')

n_rows: 17544
n_cols: 10
missing_values_per_col: {'valid_time': 0, 'u10': 0, 'v10': 0, 'fg10': 0, 'd2m': 0, 't2m': 0, 'msl': 0, 'tp': 0, 'latitude': 0, 'longitude': 0}
duplicate_rows: 0
duplicate_timestamps: 0
expected_hourly_rows: 17544
actual_rows: 17544
missing_timestamps: 0
date_range: ('2024-01-01 00:00:00', '2025-12-31 23:00:00')
unique_locations: [{'latitude': 23.25, 'longitude': 80.0}]


## Quality verdict
No missing values, no duplicate rows/timestamps, no gaps in the hourly sequence. This is expected
for ERA5 reanalysis (model-interpolated gridded data) — it is **not** representative of raw
sensor/AWS/radar feeds, which typically do have missing/faulty readings. The full fusion platform
(Phase 2+) still needs range checks, spike detection, and cross-validation against neighbouring
stations for those sources — this notebook's checks are kept general-purpose (see `quality_report`
in `src/data/load_clean.py`) so they extend to messier sources later.

In [3]:
df = clean_and_convert(raw)
df[['valid_time','t2m_c','d2m_c','msl_hpa','tp_mm','wind_speed','wind_dir_deg','relative_humidity_approx']].head()

,valid_time,t2m_c,d2m_c,msl_hpa,tp_mm,wind_speed,wind_dir_deg,relative_humidity_approx
0,2024-01-01 00:00:00,13.05460,11.61923,1016.66750,0.0,1.777238,262.475085,91.002963
1,2024-01-01 01:00:00,12.88530,11.57327,1016.97560,0.0,1.815793,275.892599,91.736536
2,2024-01-01 02:00:00,13.67098,11.96620,1017.75125,0.0,1.605895,278.256753,89.444426
3,2024-01-01 03:00:00,14.32992,12.04455,1018.32560,0.0,1.566779,285.087451,86.147000
4,2024-01-01 04:00:00,15.12643,12.47860,1018.80060,0.0,1.590144,301.524226,84.203044


## Range / physical plausibility checks

In [4]:
checks = {
    't2m_c (temperature, C)': (df['t2m_c'].min(), df['t2m_c'].max(), -10, 55),
    'msl_hpa (pressure, hPa)': (df['msl_hpa'].min(), df['msl_hpa'].max(), 870, 1085),
    'tp_mm (precip, mm/hr)': (df['tp_mm'].min(), df['tp_mm'].max(), 0, 300),
    'wind_speed (m/s)': (df['wind_speed'].min(), df['wind_speed'].max(), 0, 60),
}
for name, (lo, hi, plaus_lo, plaus_hi) in checks.items():
    ok = (lo >= plaus_lo) and (hi <= plaus_hi)
    print(f'{name}: observed [{lo:.2f}, {hi:.2f}] | plausible range [{plaus_lo}, {plaus_hi}] -> {"OK" if ok else "FLAG"}')

t2m_c (temperature, C): observed [7.04, 43.91] | plausible range [-10, 55] -> OK
msl_hpa (pressure, hPa): observed [992.25, 1022.90] | plausible range [870, 1085] -> OK
tp_mm (precip, mm/hr): observed [0.00, 20.90] | plausible range [0, 300] -> OK
wind_speed (m/s): observed [0.03, 7.95] | plausible range [0, 60] -> OK


In [5]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/jabalpur_clean.csv', index=False)
print('Saved:', df.shape)

Saved: (17544, 24)


## Summary of Notebook 02
- All values pass physical plausibility checks — no flags raised
- Converted to usable units: °C, hPa, mm/hr, m/s
- Added derived fields: wind_speed, wind_dir_deg, relative_humidity_approx, calendar fields, rain_flag
- Cleaned file saved to `data/processed/jabalpur_clean.csv`
- Next: `03_eda_visualization.ipynb`